# Foreign Whispers — Colab GPU Backend

**Instructions:**
1. **Runtime > Change runtime type → T4 GPU**
2. Run **Cell 1 (Setup)** — wait for it to finish (~3-5 min)
3. Run **Cell 2 (Server)** — copy the `API_URL` to your Mac's `.env`

In [ ]:
# ==========================================
# CELL 1: SETUP (run once per session)
# ==========================================

import os

# Clone or update the repo
if not os.path.isdir('foreign-whispers'):
    !git clone https://github.com/tilak30/foreign-whispers.git
else:
    !git -C foreign-whispers pull --rebase

%cd foreign-whispers

# System-level dependency for pyrubberband
!apt-get install -y rubberband-cli -q

# Install project into the COLAB system Python (not uv venv)
# --no-deps on torch because Colab already has the right CUDA build
!pip install -e . --no-deps -q

# Install the heavy deps that aren't pre-installed on Colab
!pip install pyngrok nest-asyncio fastapi uvicorn[standard] pydub librosa soundfile pyrubberband argostranslate transformers sentencepiece -q

print('\n✅ Setup complete — run Cell 2 to start the server')

In [ ]:
# ==========================================
# CELL 2: START SERVER + NGROK TUNNEL
# ==========================================

# Re-install pyngrok/nest-asyncio in case kernel restarted
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok', 'nest-asyncio', '-q'], check=True)

import os, time, socket
from pyngrok import ngrok
import nest_asyncio
nest_asyncio.apply()

os.environ['MPLBACKEND'] = 'Agg'

# --- PASTE YOUR NGROK TOKEN HERE ---
NGROK_TOKEN = '3D41N3dzj7hCpAfYDIXk2gdtNV1_3gBeNENvwHE32jporvsLR'
# -----------------------------------

PORT = 8080

# Kill anything already on that port
!fuser -k {PORT}/tcp 2>/dev/null || true
time.sleep(1)

# Kill any existing ngrok tunnels
ngrok.kill()
time.sleep(1)

# Start uvicorn as a background subprocess
log_file = open('/tmp/uvicorn.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.src.main:app',
     '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=log_file,
    stderr=log_file,
)

# Wait for the server to actually be listening (up to 60s)
print(f'Waiting for uvicorn on port {PORT}...', end='', flush=True)
for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            print(' ready!')
            break
    except OSError:
        print('.', end='', flush=True)
        time.sleep(2)
else:
    print('\n\n❌ Server failed to start. Check logs:')
    !cat /tmp/uvicorn.log
    raise RuntimeError('uvicorn did not bind in time')

# Open ngrok tunnel AFTER the server is confirmed up
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print()
print('=' * 60)
print('✅ YOUR COLAB GPU BACKEND IS LIVE!')
print()
print(f'  API_URL={public_url}')
print()
print('1. Add that line to your Mac .env file')
print('2. Run: docker compose --profile cpu up -d')
print('3. Delete old .wav cache, then re-run TTS in the UI')
print('=' * 60)
print()
print('📋 Server log → /tmp/uvicorn.log  (run Cell 3 to view)')
print('⚠️  Keep this cell running — stopping it kills the server.')

# Block forever so the cell (and server) stay alive
try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()
    ngrok.kill()
    print('\nServer stopped.')

In [ ]:
# ==========================================
# CELL 3 (optional): View server logs
# ==========================================
!tail -60 /tmp/uvicorn.log